# DICOM

In [ ]:
import os
import numpy as np
import pydicom
import plotly.express as px

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
read_dcm = lambda i : pydicom.dcmread(os.path.join(path_, i)).pixel_array
ct = list(map(read_dcm, sorted(os.listdir(path_))))
ct = np.stack(ct, axis=0)
print(ct.shape)

fig = px.imshow(
    ct, 
    animation_frame=0,
    binary_string=True, 
    labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
# Manufacturer (0008, 0070)
# X-Ray Tube Current (0018,1151)
# Exposure (0018,1152)
# Exposure Time (0018,1150)
# Exposure = (X-Ray Tube Current) * (Exposure Time)

import os, glob, random
import pydicom

def print_recursiverly(path, data):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                manufacturer = dcm.get((0x0008, 0x0070)).value
                xray_curr = dcm.get((0x0018, 0x1151)).value
                try:
                    exposure_t = dcm.get((0x0018, 0x1150)).value
                except (AttributeError, TypeError):
                    exposure_t = None
                try:
                    exposure = dcm.get((0x0018, 0x1152)).value
                except AttributeError:
                    exposure = None
                
                print(path)
                print(manufacturer)
                print(xray_curr)
                print(exposure_t)
                print(exposure)
                print()
                data.append({"manufacturer": manufacturer, "xray_curr": xray_curr, "exposure_t": exposure_t, 
                             "exposure": exposure, "path": path})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j, data)

data = []
N = 20

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

patients = glob.glob(os.path.join(r"C:\Users\bilel.guetarni\Desktop\data\TCIA\HNSCC-3DCT-RT\manifest-1549495779734\HNSCC-3DCT-RT", "*"))
for patient in random.sample(patients, N):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i, data)

print(len(data))

The tag (0020,0052) Frame of Reference UID is available in three different ways:
    (0020,0052) Frame of Reference UID
    (3006,0010) -> item -> (0020,0052) Frame of Reference UID
    (3006,0020) -> item -> (3006,0024) Referenced Frame of Reference UID

# RTDOSE

In [ ]:
from dicom_class import CT, RTDOSE
from dicompylercore import dicomparser, dvh, dvhcalc

ct = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000"
dose = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

In [ ]:
# i.e. Get a dict of structure information
dp = dicomparser.DicomParser(st)
structures = dp.GetStructures()
structures

In [ ]:
# Calculate a DVH from DICOM RT data
calcdvh = dvhcalc.get_dvh(st, dose, 17)
# calcdvh.relative_volume.plot()
print(calcdvh.mean)

In [ ]:
dose1 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm"
dose2 = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.5_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402276.1060.1577.dcm"
st = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm"

dp = dicomparser.DicomParser(st)
for _, str in dp.GetStructures().items():
    print(str["name"])
    calcdvh = dvhcalc.get_dvh(st, dose1, str["id"])
    print(calcdvh.mean)
    calcdvh = dvhcalc.get_dvh(st, dose2, str["id"])
    print(calcdvh.mean)

In [ ]:
from dicom_class import CT, RTDOSE, RTSTRUCT
import plotly.express as px
import numpy as np

dose = RTDOSE(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000")
rtstruct = RTSTRUCT(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")
ct = CT(
    path=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_CT_2017-07-17_133250_ORL.(sauf.sinus)_ORL.2MM.ARTIX_n200__00000",
    rtdose=dose,
    rtstruct=rtstruct)


structure_name = "C_ext-3mm"

# mask = dose.get_structure_mask(structure_name)

# fig = px.imshow(
#     np.moveaxis(mask, -1, 0), 
#     animation_frame=0,
#     binary_string=True, 
#     labels=dict(animation_frame="slice"))
# fig.show()

In [ ]:
# fig = px.imshow(
#     np.moveaxis(dose.get_voxel_array(), -1, 0), 
#     animation_frame=0,
#     binary_string=True, 
#     labels=dict(animation_frame="slice"))

dose_array = dose.get_voxel_array()
fig = px.imshow(np.moveaxis(dose.get_voxel_array(), -1, 0)[40])
fig.show()

In [ ]:
structure_name = "Parotide_homolaterale"
dose_array = dose.get_voxel_array()
dose_array[dose.get_structure_mask(structure_name)].mean()

In [ ]:
from dicompylercore import dicomparser, dvh, dvhcalc
dp = dicomparser.DicomParser(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm")
dp.GetStructures()

In [ ]:
calcdvh = dvhcalc.get_dvh(
    r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTst_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402267.473.1571.dcm", 
    r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data\002\CT5\DOE^JOHN_ANON74898_RTDOSE_2017-07-17_133250_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402274.577.1575.dcm", 
    17)
print(calcdvh.mean)

# clinical

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.mda_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[1]]
print(df["USUBJID"].unique())
print(artix.salivation_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(df["USUBJID"].unique())
print(artix.dosimetry_parsing(df))

In [ ]:
import pandas
import artix

df = pandas.read_csv(r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv", sep=";", encoding='cp1252')
df = df[df["USUBJID"] == df["USUBJID"].unique()[0]]
print(artix.aetox_parsing(df))

# print(df["USUBJID"].unique())
# print(len(df[df["USUBJID"].astype(int) == 5]))

# create ARTIX

In [ ]:
import glob, os, tqdm, random
import artix

path = r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\DICOM_ARTIX_data"
patients = []
for p in tqdm.tqdm(glob.glob(os.path.join(path, "*"))[:3], ncols=50):
    patients.append(artix.load_patient(
        path=p,
        id_map=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\ARTIX_ID_CORRELATION.xlsx",
        PATIENT_DESCRIPTION_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_PATIENT_DESCRIPTION_LTSI.csv",
        EFFICACY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_EFFICACY_LTSI.csv",
        SALIVATION_DATA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv",
        TREATMENT_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_TREATMENT_LTSI.csv",
        DOSIMETRY_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_DOSIMETRY_LTSI.csv",
        MDA_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_MDA_LTSI.csv",
        AETOXGEN_csv=r"C:\Users\bilel.guetarni\Desktop\data\ARTIX\toxicity_data\20241021_AE_TOX_GEN_LTSI.csv",
        log="./log",
        ))

print(len(patients))

In [ ]:
import pickle

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "wb") as f:
    pickle.dump(patients, f)

# plots

In [1]:
import pickle
import artix

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
# with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix - Copie.pkl", "rb") as f:
    patients = pickle.load(f)

print(len(patients))

60


In [ ]:
import pandas
import plotly.express as px
import numpy as np

map_names = {
    'PAROH_DOSE': ("ipsilateral", "parotid"),
    'PAROC_DOSE': ("contralateral", "parotid"),
    'SMAXH_DOSE': ("ipsilateral", "submandibular"),
    'SMAXC_DOSE': ("contralateral", "submandibular"),
    'MOUTH_DOSE': ("both", "mouth"),
 }

df = []
for p in patients:
    for k, v in p.clinical.items():
        if (not k in map_names.keys()) or (isinstance(v, str) and not is_float_try(v)):
            continue

        side, oar = map_names[k]
        df.append({"id": p.id, "arm": p.clinical["arm"], "side": side, "oar": oar, "dose": float(v)})
        
df = pandas.DataFrame(df)
print(df)

fig = px.box(df, x="arm", y="dose", color="side", facet_col="oar")
fig.show()

In [16]:
import pandas

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)
print(df)

print(df["type"].unique())

idx = (df["type"] == "SSF") & (df["visitID"] == 'Inclusion')
dff = df[idx]
dff["value"] = dff["value"].apply(fn)

print("total: ", len(dff))
print("xersotomia baseline unknown: ", len(dff[dff["value"].isna()]))
print("xerostomia baseline positive: ", len(dff[dff["value"] < 500]))
print("xersotomia baseline negative: ", len(dff[dff["value"] >= 500]))

      id type    value    visitID      sample
0     98  SSF     2820  Inclusion         NaN
1     98  SSF     1580         M6         NaN
2     98  SSF     1440        M12         NaN
3     98  SSF     1280        M18         NaN
4     98  SSF      980        M24         NaN
...   ..  ...      ...        ...         ...
8242  96   AE  Grade 1        M18  XEROSTOMIE
8243  96   AE  Grade 1        M21   DYSPHAGIE
8244  96   AE  Grade 1        M21  XEROSTOMIE
8245  96   AE  Grade 1        M24   DYSPHAGIE
8246  96   AE  Grade 1        M24  XEROSTOMIE

[8247 rows x 5 columns]
['SSF' 'DOSIMETRY' 'MDA' 'AE']
total:  60
xersotomia baseline unknown:  1
xerostomia baseline positive:  10
xersotomia baseline negative:  49


C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\1691620155.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [30]:
import plotly.express as px

dff = df[df["type"] == "SSF"]
dff["value"] = dff["value"].apply(fn)

# exclude basline xerostomia
inclusion_idx =  (dff["visitID"] == "Inclusion")
noxerobaseline_idx = (dff[inclusion_idx]["value"].notna()) & (dff[inclusion_idx]["value"] >= 500)
dff = dff[dff["id"].isin(dff[inclusion_idx][noxerobaseline_idx]["id"].unique())]

print(len(dff))
fig = px.line(dff, x='visitID', y='value', color='id', title="SSF(t)")
fig.add_hline(y=500, line_dash="dot")
fig.update_layout(height=400, width=800)
fig.show()

C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\327507354.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



191


In [13]:
import plotly.express as px

dff = df[df["type"] == "SSF"]
dff["value"] = dff["value"].apply(fn)

# exclude basline xerostomia
inclusion_idx =  (dff["visitID"] == "Inclusion")
noxerobaseline_idx = (dff[inclusion_idx]["value"].notna()) & (dff[inclusion_idx]["value"] >= 500)
dff = dff[dff["id"].isin(dff[inclusion_idx][noxerobaseline_idx]["id"].unique())]

no_xero_id = [id_ for id_ in dff["id"].unique() if len(dff[(dff["id"] == id_) & (dff["value"] < 500)]) == 0]
xero_id = [id_ for id_ in dff["id"].unique() if not(id_) in no_xero_id]

xero_df = dff[df["id"].isin(xero_id)]
print(len(xero_id))
fig = px.line(xero_df, x='visitID', y='value', color='id', title="SSF(t) for patients with xerostomia")
fig.add_hline(y=500, line_dash="dot")
fig.update_layout(height=400, width=800)
fig.show()

no_xero_df = dff[df["id"].isin(no_xero_id)]
print(len(no_xero_id))
fig = px.line(no_xero_df, x='visitID', y='value', color='id', title="SSF(t) for patients without xerostomia at any point in time")
fig.add_hline(y=500, line_dash="dot")
fig.update_layout(height=400, width=800)
fig.show()

27


C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\2548137524.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\2548137524.py:14: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\2548137524.py:21: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



22


In [10]:
import plotly.express as px

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

dff = df[df["type"] == "SSF"]
dff["value"] = dff["value"].apply(fn)

fig = px.histogram(dff, x="value", facet_row ="visitID")
fig.add_vline(x=500, line_dash="dot")
fig.update_layout(height=1000)
fig.show()

C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_24000\3799308175.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [3]:
import plotly.express as px

def fn(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None

dff = df[(df["type"] == "MDA") & (df["sample"] == "Q10")]
dff["value"] = dff["value"].apply(fn)

fig = px.histogram(dff, x="value", facet_row ="visitID")
fig.update_layout(height=1200)
fig.show()

C:\Users\bilel.guetarni\AppData\Local\Temp\2\ipykernel_33540\3923339511.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dff["value"] = dff["value"].apply(fn)


In [12]:
import re

def transform_SSF(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_MDA(x):
    return transform_SSF(x)
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None


# select xerostomia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE")) | (df["type"] == "SSF")]

# transform values

SSF_idx = (dff["type"] == "SSF")
dff.loc[SSF_idx, "value"] = dff[SSF_idx]["value"].apply(transform_SSF)

MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

In [5]:
t1, t2 = "AE", "MDA"

x = sorted(dff[dff["type"] == t1]["value"].dropna().unique())
y = sorted(dff[dff["type"] == t2]["value"].dropna().unique())

data = []
for xx in x:
    counts = []
    for yy in y:
        counts.append(0)
        for visit in dff["visitID"].unique():
            for id in dff["id"].unique():
                sub_dff = dff[(dff["type"].isin([t1, t2])) & (dff["visitID"] == visit) & (dff["id"] == id)]
                try:
                    if sub_dff[sub_dff["type"] == t1]["value"].item() == xx and sub_dff[sub_dff["type"] == t2]["value"].item() == yy:
                        counts[-1] = counts[-1] + 1 
                except ValueError:
                    continue
    data.append(counts)

fig = px.imshow(data,
                labels=dict(x=t2, y=t1),
                x=list(map(str, y)),
                y=list(map(str, x)),
                    text_auto=True
            )
fig.show()

In [23]:
sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)

fig = px.histogram(sub_dff, x="SSF", facet_row ="AE", marginal='violon')
fig.update_layout(height=1000)
fig.show()

sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([3,4,5]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([6,7,8]), "MDA"] = 3
sub_dff.loc[sub_dff["MDA"].isin([9]), "MDA"] = 4
fig = px.histogram(sub_dff, x="SSF", facet_row ="MDA", marginal='violon')
fig.update_layout(height=1000)
fig.show()

In [29]:
import plotly.express as px
from scipy.stats import f_oneway, spearmanr
import pandas
import pickle
import re

def transform_SSF(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_MDA(x):
    return transform_SSF(x)
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None


# select xerostomia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE")) | (df["type"] == "SSF")]

# transform values

SSF_idx = (dff["type"] == "SSF")
dff.loc[SSF_idx, "value"] = dff[SSF_idx]["value"].apply(transform_SSF)

MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)

sub_dff_bis = sub_dff[(sub_dff["SSF"].notna()) & (sub_dff["AE"].notna())]
groups = [sub_dff_bis[sub_dff_bis["AE"] == i]["SSF"].to_numpy() for i in sub_dff_bis["AE"].unique()]
print(len(groups))
print(f_oneway(*groups))
print(spearmanr(sub_dff_bis["SSF"].to_numpy(), sub_dff_bis["AE"].to_numpy()))

sub_dff_bis = sub_dff[(sub_dff["SSF"].notna()) & (sub_dff["MDA"].notna())]
# sub_dff_bis.loc[sub_dff["MDA"].isin([0,1,2,3,4]), "MDA"] = 1
# sub_dff_bis.loc[sub_dff["MDA"].isin([5,6]), "MDA"] = 2
# sub_dff_bis.loc[sub_dff["MDA"].isin([7,8,9,10]), "MDA"] = 3
groups = [sub_dff_bis[sub_dff_bis["MDA"] == i]["SSF"].to_numpy() for i in sub_dff_bis["MDA"].unique()]
print()
print(len(groups))
print(f_oneway(*groups))
print(spearmanr(sub_dff_bis["SSF"].to_numpy(), sub_dff_bis["MDA"].to_numpy()))

print()
sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))

# group MDA into smaller groups
sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([3,4,5]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([6,7,8]), "MDA"] = 3
sub_dff.loc[sub_dff["MDA"].isin([9, 10]), "MDA"] = 4
fig = px.box(sub_dff, x="MDA", y="SSF")
fig.update_layout(height=600, width=800)
fig.show()

fig = px.box(sub_dff, x="AE", y="SSF")
fig.update_layout(height=600, width=800)
fig.show()

sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)

fig = px.imshow(sub_dff_bis, text_auto=True)
fig.update_layout(height=600, width=800)
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

# # group MDA into smaller groups
# sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
# sub_dff.loc[sub_dff["MDA"].isin([3,4,5]), "MDA"] = 2
# sub_dff.loc[sub_dff["MDA"].isin([6,7,8]), "MDA"] = 3
# sub_dff.loc[sub_dff["MDA"].isin([9, 10]), "MDA"] = 4
# print(spearmanr(sub_dff["MDA"].to_numpy(), sub_dff["AE"].to_numpy()))

sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)

fig = px.imshow(sub_dff_bis, text_auto=True)
fig.update_layout(height=600, width=800)
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

2
F_onewayResult(statistic=4.12037743122714, pvalue=0.044898474737372664)
SignificanceResult(statistic=-0.16562139993941566, pvalue=0.08821790030969638)

11
F_onewayResult(statistic=2.256436158213165, pvalue=0.017029649176409156)
SignificanceResult(statistic=-0.24462989422419384, pvalue=0.0011411150746816938)

SignificanceResult(statistic=0.3908123419090549, pvalue=1.8064037849103511e-06)


In [19]:
from scipy.stats import spearmanr
import plotly.express as px
import pandas
import pickle
import re

    
def transform_MDA(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
    patients = pickle.load(f)


df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)

# select dysphagia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE"))]

# transform values
MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)
sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))


sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)

# print(sub_dff_bis)
fig = px.imshow(sub_dff_bis, text_auto=True, title="XEROSTOMIE")
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

# group MDA into smaller groups
sub_dff.loc[sub_dff["MDA"].isin([0,1,2,3,4]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([5,6,7]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([8,9,10]), "MDA"] = 3

sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))

# gather sample points
sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)
fig = px.imshow(sub_dff_bis, text_auto=True, title="XEROSTOMIE")
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

SignificanceResult(statistic=0.3908123419090549, pvalue=1.8064037849103511e-06)


SignificanceResult(statistic=0.374770895161964, pvalue=5.067189272319829e-06)


In [20]:
from scipy.stats import spearmanr
import plotly.express as px
import pandas
import pickle
import re

    
def transform_MDA(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
    patients = pickle.load(f)


df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)

# select dysphagia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q15")) | ((df["type"] == "AE") & (df["sample"] == "DYSPHAGIE"))]

# transform values
MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

sub_dff = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff.append({})
        for t in dff["type"].unique():
            try:
                sub_dff[-1].update({t: dff[(dff["type"] == t) & (dff["visitID"] == visit) & (dff["id"] == id)]["value"].item()})
            except ValueError:
                sub_dff[-1].update({t: None})

sub_dff = pandas.DataFrame(sub_dff)
sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))


sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)

# print(sub_dff_bis)
fig = px.imshow(sub_dff_bis, text_auto=True, title="DYSPHAGIE")
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

# group MDA into smaller groups
sub_dff.loc[sub_dff["MDA"].isin([0,1,2,3,4]), "MDA"] = 1
sub_dff.loc[sub_dff["MDA"].isin([5,6,7]), "MDA"] = 2
sub_dff.loc[sub_dff["MDA"].isin([8,9,10]), "MDA"] = 3

sub_dff_bis = sub_dff[(sub_dff["AE"].notna()) & (sub_dff["MDA"].notna())]
print(spearmanr(sub_dff_bis["MDA"].to_numpy(), sub_dff_bis["AE"].to_numpy()))

# gather sample points
sub_dff_bis = pandas.DataFrame()
for k1 in sub_dff["AE"].dropna().unique():
    for k2 in sub_dff["MDA"].dropna().unique():
        sub_dff_bis.loc[k1, k2] = len(sub_dff[(sub_dff["AE"] == k1) & (sub_dff["MDA"] == k2)])

sub_dff_bis = sub_dff_bis.sort_index().reindex(sorted(sub_dff_bis.columns), axis=1)
fig = px.imshow(sub_dff_bis, text_auto=True, title="DYSPHAGIE")
fig.update_layout(xaxis_title=dict(text='MDA'), yaxis_title=dict(text='CTCAE'))
fig.show()

SignificanceResult(statistic=0.3467675944754, pvalue=0.022720684111807755)


SignificanceResult(statistic=0.2604857690203585, pvalue=0.0915937370328099)


In [134]:
import plotly.express as px
import pandas
import pickle
import re

    
def transform_SSF(x):
    if isinstance(x, str) and x.isdigit():
        return int(x)
    else:
        return None
    
def transform_MDA(x):
    return transform_SSF(x)
    
def transform_AE(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

with open(r"C:\Users\bilel.guetarni\Desktop\tmp\artix.pkl", "rb") as f:
    patients = pickle.load(f)


df = []
for p in patients:
    for cm in p.clinical_measurements:
        df.append({"id": p.id, **cm})

df = pandas.DataFrame(df)

# select dysphagia CTCAE and MDA
dff = df[((df["type"] == "MDA") & (df["sample"] == "Q10")) | ((df["type"] == "AE") & (df["sample"] == "XEROSTOMIE")) | (df["type"] == "SSF")]

# transform values

SSF_idx = (dff["type"] == "SSF")
dff.loc[SSF_idx, "value"] = dff[SSF_idx]["value"].apply(transform_SSF)

MDA_idx = (dff["type"] == "MDA")
dff.loc[MDA_idx,"value"] = dff[MDA_idx]["value"].apply(transform_MDA)

AE_idx = (dff["type"] == "AE")
dff.loc[AE_idx,"value"] = dff[AE_idx]["value"].apply(transform_AE)

# change AE visitID=S0 into Inclusion
dff.loc[(dff["type"] == "AE") & (dff["visitID"] == "S0"), "visitID"] = "Inclusion"

data = []
for visit in dff["visitID"].unique():
    for id in dff["id"].unique():
        sub_dff = dff[(dff["visitID"] == visit) & (dff["id"] == id)]
        try:
            if sub_dff[sub_dff["type"] == "AE"]["value"].notna().item() and sub_dff[sub_dff["type"] == "MDA"]["value"].notna().item() and sub_dff[sub_dff["type"] == "SSF"]["value"].notna().item():
                data.append({"AE": sub_dff[sub_dff["type"] == "AE"]["value"].item(), "MDA": sub_dff[sub_dff["type"] == "MDA"]["value"].item(), "SSF": sub_dff[sub_dff["type"] == "SSF"]["value"].item()})
        except ValueError:
            continue

sub_dff = pandas.DataFrame(data)
print(sub_dff)

# sub_dff.loc[sub_dff["MDA"].isin([0,1]), "MDA"] = 1
# sub_dff.loc[sub_dff["MDA"].isin([2,3,4]), "MDA"] = 2
# sub_dff.loc[sub_dff["MDA"].isin([5,6,7]), "MDA"] = 3
# sub_dff.loc[sub_dff["MDA"].isin([8,9,10]), "MDA"] = 4

# sub_dff.loc[sub_dff["MDA"].isin([0,1,2]), "MDA"] = 1
# sub_dff.loc[sub_dff["MDA"].isin([3,4,5,6,7]), "MDA"] = 2
# sub_dff.loc[sub_dff["MDA"].isin([8,9,10]), "MDA"] = 3


# data = []
# for i in sub_dff["MDA"].unique():
#     for j in sub_dff["AE"].unique():
#         ssf_ = sub_dff[(sub_dff["MDA"] == i) & (sub_dff["AE"] == j)]["SSF"]
#         data.append({"MDA": i, "AE": j, "mean": ssf_.mean(), "std": ssf_.std(ddof=0), "N": len(ssf_)})

# sub_dff = pandas.DataFrame(data)
# print(sub_dff)

    AE  MDA     SSF
0    1  2.0  2820.0
1    1  2.0  1580.0
2    2  6.0   440.0
3    1  2.0   740.0
4    2  8.0   480.0
..  ..  ...     ...
95   1  1.0  1440.0
96   2  8.0   980.0
97   1  5.0   200.0
98   1  2.0   480.0
99   1  3.0   320.0

[100 rows x 3 columns]


In [156]:
from sklearn.linear_model import LinearRegression
X, y = np.array(list(zip(sub_dff["MDA"].to_numpy(), sub_dff["AE"].to_numpy()))), sub_dff["SSF"].to_numpy() / sub_dff["SSF"].max()
clf = LinearRegression().fit(X, y)
print(clf.score(X,y))

0.04347588343637099


# CTCAE